# 03 Margem de erro e testes

Este notebook responde a pergunta do projeto com margem de erro e testes: em quais áreas e em quais dimensões (renda, escolaridade, raça e região) a diferença em resolver serviços públicos pela internet é real, e de que tamanho ela é. No 02, os % não tinham margem de erro. Aqui, cada % sai com o seu intervalo de confiança (IC), e as diferenças passam por um teste.

- **Entra:** `data/processed/dados_harmonizados.csv` e `data/processed/individuos_pesos_replica.csv`, gerados pelo notebook 01, e as planilhas oficiais da CETIC em `docs/fontes_cetic/tabelas/` (só na conferência da seção 4)
- **Sai:** em `outputs/`, as tabelas `estimativas_ic.csv`, `gaps_ic.csv` e `testes_rao_scott.csv`, pro Power BI

**Como ler:** cada seção começa com um texto explicando o porquê, e cada célula de código tem comentários. Rodar sempre com **Restart + Run All**, de cima pra baixo. Algumas células mudam variáveis (por exemplo, o filtro do universo e as colunas novas), então rodar uma célula de novo, fora da ordem, pode dar resultado errado.

**Roteiro**
1. Preparar o ambiente
2. Carregar os dados e escolher o universo
3. A margem de erro com as 200 réplicas
4. Conferência com as tabelas oficiais da CETIC
5. Uma função pra qualquer grupo
6. As faixas da análise
7. O tabelão com IC
8. Os gaps com IC
9. O teste de Rao-Scott
10. A correção de Holm
11. O tamanho da diferença: V de Cramér
12. O ranking das áreas
13. Testes de integridade
14. Salvar
15. Resumo dos achados

## 1. Preparar o ambiente

- `pandas` e `numpy`: as tabelas e as contas de sempre.
- `polars`: outra biblioteca de tabelas, parecida com o pandas. O `svy` só aceita ela.
- `svy`: a biblioteca do teste de Rao-Scott (seção 9), fixada na versão 0.31.0 no `requirements.txt`.
- `association`, do `scipy`: calcula o V de Cramér (seção 11).
- `encontrar_raiz_projeto` e `porcentagem_com_peso` são as mesmas dos notebooks 01 e 02.

In [1]:
# Bibliotecas
import os                                          # monta caminhos de arquivo (pastas e nomes)
import numpy as np                                 # contas com listas de números (usado no Holm)
import pandas as pd                                # tabelas (DataFrame) e colunas (Series)
import polars as pl                                # outra biblioteca de tabelas: o svy só aceita ela
import svy                                         # o teste de Rao-Scott
from scipy.stats import chi2_contingency           # o qui-quadrado comum (só pra comparar, seção 9)
from scipy.stats.contingency import association    # o V de Cramér


def encontrar_raiz_projeto(pasta_atual):
    # Sobe de pasta em pasta até achar a que tem src/harmonizacao.py (igual aos notebooks 01 e 02)
    pasta = os.path.abspath(pasta_atual)
    while True:
        if os.path.exists(os.path.join(pasta, 'src', 'harmonizacao.py')):
            return pasta
        pasta_pai = os.path.dirname(pasta)
        if pasta_pai == pasta:
            raise FileNotFoundError('Não achei src/harmonizacao.py subindo a partir de ' + pasta_atual)
        pasta = pasta_pai


raiz_projeto = encontrar_raiz_projeto(os.getcwd())
pd.set_option('display.max_columns', None)


def porcentagem_com_peso(resposta, peso):
    # % de 1 numa coluna 0/1, usando o peso de cada pessoa e ignorando os vazios (NaN)
    tem_resposta = resposta.notna()
    y = resposta[tem_resposta]
    w = peso[tem_resposta]
    return 100 * (w * y).sum() / w.sum()

## 2. Carregar os dados e escolher o universo

- O notebook usa os dois arquivos do 01: `dados_harmonizados` (uma linha por pessoa, com os dados da casa dela) e `individuos_pesos_replica` (as 200 réplicas de cada pessoa).
- O `merge` junta os dois pela chave da pessoa (`chave_domicilio`, `ID_MORADOR` e `ano`). O `validate='one_to_one'` garante que cada pessoa aparece uma vez só em cada arquivo, e o `indicator=True` confere que todo mundo achou as suas réplicas.
- O `.copy()` nas réplicas evita um aviso de desempenho do pandas 3 (o mesmo do notebook 01).
- **O universo:** só os usuários de Internet de 16 anos ou mais (`usuario_internet == 1`), igual às tabelas da CETIC e ao notebook 02 (decisão de 12/10, no caderno de bordo).

In [2]:
# Os dois arquivos que saem do notebook 01
dados = pd.read_csv(os.path.join(raiz_projeto, 'data', 'processed', 'dados_harmonizados.csv'))
replicas = pd.read_csv(os.path.join(raiz_projeto, 'data', 'processed', 'individuos_pesos_replica.csv'))

# Confere se são os arquivos certos (se der erro, rode o notebook 01 de novo)
assert dados.shape == (66976, 33), f'Tamanho inesperado: {dados.shape}. Rode o notebook 01 de novo.'
assert replicas.shape == (66976, 204), f'Tamanho inesperado: {replicas.shape}. Rode o notebook 01 de novo.'

chaves = ['chave_domicilio', 'ID_MORADOR', 'ano']       # o que identifica uma pessoa
colunas_rep = [f'rep{r}' for r in range(1, 201)]        # 'rep1', 'rep2', ..., 'rep200'

# Junta as 200 réplicas em cada pessoa
base = dados.merge(replicas[chaves + colunas_rep].copy(), on=chaves, how='left',
                   validate='one_to_one', indicator=True)
assert (base['_merge'] == 'both').all(), 'Alguma pessoa não achou as suas réplicas'
base = base.drop(columns='_merge')            # a coluna de conferência não precisa mais

# O universo da CETIC: usuários de Internet de 16 anos ou mais
base = base[base['usuario_internet'] == 1]
assert len(base) == 13859 + 14686 + 16891, 'O universo não tem o tamanho esperado'
print(base.shape)

(45436, 233)


## 3. A margem de erro com as 200 réplicas

**A ideia.** Imagine que a CETIC repetisse a pesquisa 200 vezes, sorteando outros grupos de casas. Cada réplica é uma dessas repetições "de mentirinha": um peso diferente pra cada pessoa. Se o % dá quase igual nas 200, dá pra confiar nele. Se pula muito, não.

- **EP (erro padrão):** o desvio padrão das 200 estimativas, ou seja, o quanto o % balança.
- **Margem:** 1,96 × EP (95% de confiança).
- **IC (intervalo de confiança):** de "% menos a margem" até "% mais a margem". É onde o valor de verdade provavelmente está.
- **CV:** o EP dividido pelo %. Até 30%, o número é confiável; acima disso, é fraco (decisão nº 4).

O exemplo é saúde 2024: o G2 (resolveu tudo pela internet), entre quem tentou. Nesta seção as margens ainda saem sem o fator da decisão nº 9 (seção 4). Da seção 5 em diante, todas usam o fator.

In [3]:
# Saúde 2024: só quem respondeu o G2 de saúde (quem tentou)
d24 = base[base['ano'] == 2024]
saude = d24[d24['G2_B'].notna()]

# A estimativa normal, com o peso principal
estimativa = porcentagem_com_peso(saude['G2_B'], saude['peso_amostral_individuo'])

# A mesma conta 200 vezes, cada vez usando uma réplica no lugar do peso
estimativas_rep = pd.Series([porcentagem_com_peso(saude['G2_B'], saude[coluna]) for coluna in colunas_rep])

print('Estimativa (peso principal):', round(estimativa, 1))
print('As 200 réplicas vão de', round(estimativas_rep.min(), 1), 'a', round(estimativas_rep.max(), 1))

Estimativa (peso principal): 25.8
As 200 réplicas vão de 18.7 a 35.3


In [4]:
erro_padrao = estimativas_rep.std()          # desvio padrão das 200 porcentagens
margem = 1.96 * erro_padrao                  # 95% de confiança

print('Estimativa:', round(estimativa, 1))
print('Erro padrão:', round(erro_padrao, 2))
print('Margem de erro: ±', round(margem, 1))
print('Intervalo de confiança: de', round(estimativa - margem, 1), 'a', round(estimativa + margem, 1))
print('CV (%):', round(100 * erro_padrao / estimativa, 1))

Estimativa: 25.8
Erro padrão: 2.83
Margem de erro: ± 5.5
Intervalo de confiança: de 20.3 a 31.4
CV (%): 11.0


In [5]:
# Um grupo pequeno: "não tem renda" (código 7) em saúde 2024
grupo = saude[saude['renda_familiar'] == 7]
estimativa_grupo = porcentagem_com_peso(grupo['G2_B'], grupo['peso_amostral_individuo'])
reps_grupo = pd.Series([porcentagem_com_peso(grupo['G2_B'], grupo[coluna]) for coluna in colunas_rep])
erro_grupo = reps_grupo.std()
margem_grupo = 1.96 * erro_grupo

print('Pessoas:', len(grupo), '| estimativa:', round(estimativa_grupo, 1), '| margem: ±', round(margem_grupo, 1),
      '| IC: de', round(estimativa_grupo - margem_grupo, 1), 'a', round(estimativa_grupo + margem_grupo, 1),
      '| CV:', round(100 * erro_grupo / estimativa_grupo, 1))

Pessoas: 51 | estimativa: 5.9 | margem: ± 12.7 | IC: de -6.8 a 18.5 | CV: 110.3


O que dá pra ver:
- **Saúde 2024: 25,8%, com IC de 20,3 a 31,4 e CV de 11,0.** As réplicas vão de 18,7 a 35,3, mas a maioria fica perto do centro. É um número confiável.
- **"Não tem renda": IC de −6,8 a 18,5 e CV de 110.** Não existe % negativo: é a conta dizendo que o grupo (51 pessoas, com pesos muito desiguais) é pequeno demais pra dizer quase nada. Por isso a decisão nº 2 junta esse grupo com "até 1 SM" (seção 6).

## 4. Conferência com as tabelas oficiais da CETIC

Antes de usar as réplicas nos testes, a gente conferiu se os nossos números batem com os oficiais. As planilhas estão em `docs/fontes_cetic/tabelas/`, baixadas do site da CETIC. Na aba "G2", a linha 4 é o Total (Brasil) e as colunas 2 a 8 são as 7 áreas do nosso G1 (procurou ou realizou o serviço), na ordem A a G, entre os usuários de Internet de 16 anos ou mais.

### 4.1 Os %

- No começo, os % não batiam. O motivo era o universo: o questionário aplica o módulo G a quem usou a internet nos últimos 3 meses pela C3 **ou** pela J3 (no celular), mas as tabelas oficiais contam só a C3. Isso já está resolvido na seção 2.
- A diferença que sobra (no máximo 0,07 ponto) vem do "não sabe / não respondeu" do G1. No nosso arquivo ele é NaN e fica fora da conta; a CETIC conta como "não". Contando do mesmo jeito (`G1 == 1` vira 1, e todo o resto, inclusive o NaN, vira 0), bate exatamente.
- Isso só importa nesta conferência. A medida do projeto é o G2 entre quem tentou, e o NS/NR do G2 fica de fora (como no 02).

In [6]:
# Planilha oficial de proporções de 2024, aba "G2" (o nosso G1)
caminho = os.path.join(raiz_projeto, 'docs', 'fontes_cetic', 'tabelas',
                       'tic_domicilios_2024_individuos_tabela_proporcao_v1.1.xlsx')
oficial = pd.read_excel(caminho, sheet_name='G2', header=None)   # header=None: lê tudo, sem adivinhar o cabeçalho
oficial.iloc[:6, :4]                                              # o começo da planilha

,0,1,2,3
0,"G2 - USUÁRIOS DE INTERNET, POR TIPO DE INFORMA...",NaN,NaN,NaN
1,Total de usuários de Internet com 16 anos ou mais,NaN,NaN,NaN
2,Percentual (%),NaN,"Documentos pessoais, como RG, CPF, passaporte ...","Saúde pública, como agendamento de consultas, ..."
3,NaN,NaN,"Documentos pessoais, como RG, CPF, passaporte ...","Saúde pública, como agendamento de consultas, ..."
4,TOTAL,Total,30.8547,31.8147
5,ÁREA,Urbana,32.0975,33.2772


In [7]:
# Na planilha, a linha 4 é o Total (Brasil) e as colunas 2 a 8 são as 7 áreas, na ordem A a G
oficial_2024 = oficial.iloc[4, 2:9].astype(float).round(2).tolist()

d24 = base[base['ano'] == 2024]
nosso, nosso_ns_nao = [], []
for letra in 'ABCDEFG':
    g1 = d24[f'G1_{letra}']
    nosso.append(round(porcentagem_com_peso(g1, d24['peso_amostral_individuo']), 2))             # o NaN fica de fora
    g1_ns_nao = (g1 == 1).astype(float)                                                            # o NaN vira 0, como a CETIC faz
    nosso_ns_nao.append(round(porcentagem_com_peso(g1_ns_nao, d24['peso_amostral_individuo']), 2))

comparacao = pd.DataFrame({'area': list('ABCDEFG'), 'cetic': oficial_2024, 'nosso': nosso, 'nosso_ns_nao': nosso_ns_nao})
comparacao['diferenca'] = (comparacao['nosso_ns_nao'] - comparacao['cetic']).round(2)
comparacao

,area,cetic,nosso,nosso_ns_nao,diferenca
0,A,30.85,30.90,30.85,0.0
1,B,31.81,31.85,31.81,0.0
2,C,21.78,21.79,21.78,0.0
3,D,25.28,25.31,25.28,0.0
4,E,28.72,28.79,28.72,0.0
5,F,9.26,9.27,9.26,0.0
6,G,14.27,14.30,14.27,0.0


### 4.2 As margens de erro

- A margem oficial está no arquivo de margem de erro, na mesma aba. A nossa é 1,96 × o desvio padrão das 200 réplicas, com o NS/NR contado como "não" (igual ao 4.1). A tabela mostra a razão: quantas vezes a margem oficial é maior que a nossa.
- **2024 e 2025:** a oficial é a nossa × 1,054 e × 1,14, igual em todas as áreas. É um fator constante que o relatório da CETIC não explica (são as raízes de 10/9 e de 1,3). **Decisão nº 9:** a gente aplica o mesmo fator nas nossas margens (seção 5), pra ficarem iguais às oficiais. É o lado mais cauteloso: com margem maior, fica mais difícil achar uma diferença que não existe.
- **2023:** a razão fica perto de 1 (saúde é o caso mais longe, cerca de 1,25). Em 2023 a CETIC calculou a margem por outro método (o do conglomerado primário), então não dá pra bater exato. 2023 fica sem fator.
- Foi esta conferência que mostrou que, no arquivo bruto de 2023, as réplicas eram contagens do sorteio (0 a 5) e não pesos. O notebook 01 já multiplica pelo `PESO` (seção 4.1 do 01).
- O `enumerate` dá a posição (0, 1, 2...) junto com a letra, pra pegar a margem oficial da mesma área.

In [8]:
versao = {2023: 'v1.1', 2024: 'v1.1', 2025: 'v1.0'}     # a versão da planilha muda por ano

linhas = []
for ano in [2023, 2024, 2025]:
    caminho = os.path.join(raiz_projeto, 'docs', 'fontes_cetic', 'tabelas',
                           f'tic_domicilios_{ano}_individuos_tabela_margem_de_erro_{versao[ano]}.xlsx')
    margem_oficial = pd.read_excel(caminho, sheet_name='G2', header=None).iloc[4, 2:9].astype(float).tolist()

    d = base[base['ano'] == ano]
    for i, letra in enumerate('ABCDEFG'):
        g1 = (d[f'G1_{letra}'] == 1).astype(float)                                   # NS/NR como "não", igual à CETIC
        reps = pd.Series([porcentagem_com_peso(g1, d[coluna]) for coluna in colunas_rep])
        nossa = 1.96 * reps.std()
        linhas.append({'ano': ano, 'area': letra, 'cetic': round(margem_oficial[i], 2), 'nossa': round(nossa, 2),
                       'razao': round(margem_oficial[i] / nossa, 3)})

margens = pd.DataFrame(linhas)
margens.pivot_table(index='area', columns='ano', values='razao')     # quantas vezes a margem da CETIC é maior que a nossa

ano,2023,2024,2025
area,,,
A,1.003,1.054,1.141
B,1.255,1.054,1.141
C,1.000,1.054,1.141
D,1.033,1.054,1.141
E,1.015,1.054,1.141
F,0.985,1.054,1.141
G,1.003,1.054,1.141


## 5. Uma função pra qualquer grupo

Daqui pra frente, a mesma conta vai ser feita centenas de vezes, então ela vira uma função. Ela também fica mais rápida: em vez de chamar `porcentagem_com_peso` 200 vezes, faz as 200 réplicas numa linha só.

Primeiro, um exemplo de brinquedo, com 3 pessoas e 2 réplicas, pra ver essa conta funcionando e conferir de cabeça:
- `.mul(..., axis=0)` multiplica cada **linha** (cada pessoa) pelo 0 ou 1 dela. Quem não resolveu perde o peso.
- `.sum()` soma cada coluna (cada réplica). Dividindo pela soma de todos os pesos, sai um % por réplica.

In [9]:
# Exemplo de brinquedo: 3 pessoas e 2 réplicas
mini = pd.DataFrame({
    'resolveu': [1, 0, 1],          # o G2 de cada pessoa (1 = resolveu)
    'rep1':     [10, 20, 30],       # o peso de cada pessoa na réplica 1
    'rep2':     [30, 10, 20],       # o peso de cada pessoa na réplica 2
})

# Multiplica cada linha pelo 0 ou 1 da pessoa: quem não resolveu perde o peso
pesos_de_quem_resolveu = mini[['rep1', 'rep2']].mul(mini['resolveu'], axis=0)
print(pesos_de_quem_resolveu, '\n')

# Peso de quem resolveu ÷ peso de todo mundo, em cada réplica
pct_por_replica = 100 * pesos_de_quem_resolveu.sum() / mini[['rep1', 'rep2']].sum()
print(pct_por_replica.round(1), '\n')

print('Quanto o % balança entre as réplicas (EP):', round(pct_por_replica.std(), 1))

   rep1  rep2
0    10    30
1     0     0
2    30    20 

rep1    66.7
rep2    83.3
dtype: float64 

Quanto o % balança entre as réplicas (EP): 11.8


A função faz a mesma conta com as 200 réplicas e já aplica o fator da decisão nº 9 (seção 4.2). Ela devolve um dicionário, que depois vira uma linha de tabela: `n`, `pct`, `ep`, `ic_inf`, `ic_sup` e `cv`.

O teste é saúde 2024, a mesma conta da seção 3. O EP passa de 2,83 pra 2,98 (2,83 × 1,054).

In [10]:
# Ajuste da decisão nº 9: deixa a nossa margem igual à da CETIC (2023 fica sem ajuste)
fator = {2023: 1.0, 2024: 1.054, 2025: 1.140}


def estimativa_com_ic(responderam, coluna_area, ano):
    # responderam: as pessoas de um grupo que responderam o G2 da área (sem NaN)
    y = responderam[coluna_area]                                # 1 = resolveu, 0 = não
    peso = responderam['peso_amostral_individuo']
    pct = 100 * (peso * y).sum() / peso.sum()                   # o % de sempre

    reps = responderam[colunas_rep]                             # as 200 réplicas
    pct_reps = 100 * reps.mul(y, axis=0).sum() / reps.sum()     # a conta do brinquedo: um % por réplica
    ep = pct_reps.std() * fator[ano]                            # quanto o % balança (com o ajuste)
    margem = 1.96 * ep

    return {'n': len(responderam), 'pct': round(float(pct), 1), 'ep': round(float(ep), 2),
            'ic_inf': round(float(pct - margem), 1), 'ic_sup': round(float(pct + margem), 1),
            'cv': round(float(100 * ep / pct), 1)}


# Teste: saúde 2024
d24 = base[base['ano'] == 2024]
saude = d24[d24['G2_B'].notna()]          # só quem respondeu o G2 de saúde
estimativa_com_ic(saude, 'G2_B', 2024)

{'n': 3892,
 'pct': 25.8,
 'ep': 2.98,
 'ic_inf': 20.0,
 'ic_sup': 31.7,
 'cv': 11.5}

## 6. As faixas da análise

A decisão nº 2 junta as faixas que são pequenas demais (o 02 mostrou isso com o n efetivo):
- **Renda em 5:** "não tem renda" (7) junto com "até 1 SM" (1), e "mais de 10 SM" (6) junto com "5 a 10 SM" (5), formando "mais de 5 SM".
- **Escolaridade em 3:** "analfabeto/infantil" (1) junto com "fundamental" (2), formando "até fundamental".
- Raça e região ficam como estão.

O `.map` troca cada código antigo pelo novo, seguindo o dicionário. A conferência mostra que cada faixa antiga (linha) cai numa faixa nova só (coluna).

In [11]:
# Decisão nº 2: as faixas da análise
mapa_renda5 = {1: 1, 7: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 5}
nomes_renda5 = {1: 'até 1 SM', 2: '1 a 2 SM', 3: '2 a 3 SM', 4: '3 a 5 SM', 5: 'mais de 5 SM'}

mapa_escolaridade3 = {1: 1, 2: 1, 3: 2, 4: 3}
nomes_escolaridade3 = {1: 'até fundamental', 2: 'médio', 3: 'superior'}

# Cria as colunas novas: o .map troca cada código antigo pelo novo
base['renda5'] = base['renda_familiar'].map(mapa_renda5)
base['escolaridade3'] = base['escolaridade'].map(mapa_escolaridade3)

# Confere: cada faixa antiga (linha) tem que cair numa faixa nova só (coluna)
print(pd.crosstab(base['renda_familiar'], base['renda5']), '\n')
print(pd.crosstab(base['escolaridade'], base['escolaridade3']))

renda5            1.0    2.0   3.0   4.0   5.0
renda_familiar                                
1.0             15437      0     0     0     0
2.0                 0  12599     0     0     0
3.0                 0      0  6660     0     0
4.0                 0      0     0  4264     0
5.0                 0      0     0     0  2027
6.0                 0      0     0     0   730
7.0               701      0     0     0     0 

escolaridade3      1      2     3
escolaridade                     
1               1828      0     0
2              14484      0     0
3                  0  21387     0
4                  0      0  7737


Por que juntar: o mesmo cálculo antes e depois, nas duas pontas da renda, em saúde 2024. A `saude` é recriada porque a antiga foi feita antes das colunas novas existirem.

In [12]:
saude = base[(base['ano'] == 2024) & base['G2_B'].notna()]       # recria, agora com as colunas novas

exemplos = {
    'não tem renda (sozinho)': saude['renda_familiar'] == 7,
    'até 1 SM (junto com não tem renda)': saude['renda5'] == 1,
    'mais de 10 SM (sozinho)': saude['renda_familiar'] == 6,
    'mais de 5 SM (5 a 10 e mais de 10 juntas)': saude['renda5'] == 5,
}
linhas = []
for nome, filtro in exemplos.items():
    linha = estimativa_com_ic(saude[filtro], 'G2_B', 2024)
    linha['grupo'] = nome
    linhas.append(linha)

pd.DataFrame(linhas).set_index('grupo')

,n,pct,ep,ic_inf,ic_sup,cv
grupo,,,,,,
não tem renda (sozinho),51,5.9,6.82,-7.5,19.2,116.3
até 1 SM (junto com não tem renda),1192,23.5,3.23,17.1,29.8,13.8
mais de 10 SM (sozinho),56,37.8,12.96,12.4,63.2,34.3
mais de 5 SM (5 a 10 e mais de 10 juntas),260,34.6,9.75,15.5,53.7,28.2


O que dá pra ver:
- **"Não tem renda" sozinho:** IC de −7,5 a 19,2 e CV de 116. Junto com "até 1 SM", o IC vai de 17,1 a 29,8 e o CV cai pra 13,8.
- **A ponta de cima:** o CV cai de 34,3 ("mais de 10 SM") pra 28,2 ("mais de 5 SM") e fica abaixo de 30. Mas o IC ainda é largo (de 15,5 a 53,7): juntar ajuda, mas a amostra tem pouca gente rica.

## 7. O tabelão com IC

A mesma conta pra todos os anos, áreas e grupos, numa tabela só. São 4 loops, um dentro do outro: ano → área → dimensão (renda, escolaridade, raça, região) → categoria. Cada volta do último vira uma linha.

- **Tamanho previsto:** 22 combinações de ano e área (7 áreas × 3 anos, mais justiça em 2025) × 19 categorias (1 Brasil + 5 de renda + 3 de escolaridade + 5 de raça + 5 de região) = **418 linhas**.
- **O truque do `total`:** uma coluna que vale 1 pra todo mundo. O "grupo" dela é o Brasil inteiro, e o mesmo loop faz a linha do total sem código a mais.
- **`aviso`:** True quando o CV passa de 30% (decisão nº 4). Amarela e indígena ficam no tabelão, com o IC e o aviso, mas fora dos testes (seção 9).
- **`linha.update(...)`:** a função devolve n, pct, ep, IC e CV, e o `update` junta de onde a linha veio (ano, área, dimensão e categoria).

In [13]:
# Truque: uma coluna que vale 1 pra todo mundo. O "grupo" dela é o Brasil inteiro
base['total'] = 1

# Cada dimensão anda junto com o seu dicionário de nomes (faixas novas da decisão nº 2)
grupos = {
    'total': {1: 'Brasil'},
    'renda5': nomes_renda5,
    'escolaridade3': nomes_escolaridade3,
    'raca': {1: 'branca', 2: 'preta', 3: 'parda', 4: 'amarela', 5: 'indígena'},
    'regiao': {1: 'Sudeste', 2: 'Nordeste', 3: 'Sul', 4: 'Norte', 5: 'Centro-Oeste'},
}
nomes_areas = {'A': 'documentos', 'B': 'saúde', 'C': 'educação', 'D': 'trabalho',
               'E': 'impostos', 'F': 'polícia', 'G': 'transporte', 'H': 'justiça'}

In [14]:
linhas = []
for ano in [2023, 2024, 2025]:
    d = base[base['ano'] == ano]                                       # um ano
    for letra, nome_area in nomes_areas.items():
        if letra == 'H' and ano != 2025:                               # justiça só existe em 2025
            continue
        responderam = d[d[f'G2_{letra}'].notna()]                      # quem respondeu o G2 dessa área
        for coluna_grupo, nomes in grupos.items():
            for codigo, nome in nomes.items():
                grupo = responderam[responderam[coluna_grupo] == codigo]     # uma categoria
                linha = estimativa_com_ic(grupo, f'G2_{letra}', ano)         # n, pct, ep, IC e CV
                linha.update({'ano': ano, 'area': nome_area, 'dimensao': coluna_grupo, 'categoria': nome})
                linhas.append(linha)

tabelao = pd.DataFrame(linhas)
tabelao['aviso'] = tabelao['cv'] > 30                                  # decisão nº 4: CV acima de 30% = número fraco
tabelao = tabelao[['ano', 'area', 'dimensao', 'categoria', 'n', 'pct', 'ep', 'ic_inf', 'ic_sup', 'cv', 'aviso']]
print(tabelao.shape)
tabelao.head(19)

(418, 11)


,ano,area,dimensao,categoria,n,pct,ep,ic_inf,ic_sup,cv,aviso
0,2023,documentos,total,Brasil,3629,35.2,4.15,27.1,43.4,11.8,False
1,2023,documentos,renda5,até 1 SM,974,18.2,3.23,11.8,24.5,17.8,False
2,2023,documentos,renda5,1 a 2 SM,988,41.0,10.02,21.4,60.6,24.4,False
3,2023,documentos,renda5,2 a 3 SM,635,49.3,6.10,37.3,61.2,12.4,False
4,2023,documentos,renda5,3 a 5 SM,492,29.8,5.40,19.2,40.4,18.1,False
5,2023,documentos,renda5,mais de 5 SM,351,33.1,6.06,21.3,45.0,18.3,False
6,2023,documentos,escolaridade3,até fundamental,708,30.0,6.29,17.7,42.3,21.0,False
7,2023,documentos,escolaridade3,médio,1961,35.8,3.99,27.9,43.6,11.2,False
8,2023,documentos,escolaridade3,superior,960,37.3,7.70,22.2,52.3,20.7,False
9,2023,documentos,raca,branca,987,38.1,9.02,20.5,55.8,23.6,False


In [15]:
# 22 combinações de ano e área × 19 categorias = 418 linhas
assert tabelao.shape == (418, 11), tabelao.shape
assert (tabelao['n'] > 0).all(), 'Tem categoria vazia'

# A linha do Brasil em saúde 2024 tem que ser igual ao teste da função (seção 5)
print(tabelao[(tabelao['ano'] == 2024) & (tabelao['area'] == 'saúde') & (tabelao['categoria'] == 'Brasil')], '\n')

# Quantas linhas têm aviso, e de quais categorias
print(tabelao['aviso'].sum(), 'de', len(tabelao), 'linhas com CV acima de 30%')
print(tabelao[tabelao['aviso']]['categoria'].value_counts())

      ano   area dimensao categoria     n   pct    ep  ic_inf  ic_sup    cv  \
152  2024  saúde    total    Brasil  3892  25.8  2.98    20.0    31.7  11.5   

     aviso  
152  False   

52 de 418 linhas com CV acima de 30%
categoria
indígena           21
amarela            13
preta               7
até fundamental     5
1 a 2 SM            2
Sudeste             1
2 a 3 SM            1
mais de 5 SM        1
superior            1
Name: count, dtype: int64


In [16]:
# As linhas em que o IC começa abaixo de zero (um % impossível)
tabelao[tabelao['ic_inf'] < 0]

,ano,area,dimensao,categoria,n,pct,ep,ic_inf,ic_sup,cv,aviso
13,2023,documentos,raca,indígena,58,7.3,5.80,-4.0,18.7,79.2,True
32,2023,saúde,raca,indígena,67,4.4,4.57,-4.6,13.4,104.2,True
51,2023,educação,raca,indígena,49,14.1,11.39,-8.2,36.4,80.7,True
108,2023,polícia,raca,indígena,29,31.4,25.38,-18.3,81.2,80.7,True
127,2023,transporte,raca,indígena,29,15.9,8.64,-1.0,32.8,54.4,True
184,2024,educação,raca,indígena,59,18.8,11.15,-3.1,40.6,59.4,True
203,2024,trabalho,raca,indígena,59,19.8,13.92,-7.5,47.1,70.3,True
238,2024,polícia,raca,preta,178,19.3,17.45,-14.9,53.5,90.3,True
241,2024,polícia,raca,indígena,25,28.3,15.11,-1.3,57.9,53.5,True
260,2024,transporte,raca,indígena,23,9.7,8.95,-7.9,27.2,92.5,True


O que dá pra ver:
- **52 das 418 linhas têm aviso**, quase todas de raça: indígena (21), amarela (13) e preta (7). Por isso a decisão nº 4 tira amarela e indígena dos testes e mostra as duas à parte.
- **15 linhas têm o IC começando abaixo de zero**, e todas têm aviso: o aviso do CV pega todos os casos impossíveis. Uma delas (amarela, transporte 2025) ainda passa de 100% no outro lado (102,7).

## 8. Os gaps com IC

O gap é a diferença entre dois grupos, por exemplo "mais de 5 SM" menos "até 1 SM", em pontos percentuais (p.p.). A pergunta é se ela existe de verdade ou se pode ser só o balanço da amostra.

- **A conta:** o gap é calculado dentro de cada uma das 200 réplicas. O desvio padrão desses 200 gaps é o EP do gap, e daí sai o IC.
- **`diferenca_real`:** True quando o IC do gap fica inteiro de um lado do zero. Se ele passa pelo zero, não dá pra dizer que existe diferença.
- **Por que não comparar os ICs dos dois grupos:** eles podem se cruzar e a diferença ainda ser real. O certo é o IC do próprio gap. Fazer a diferença réplica por réplica já leva em conta que os dois grupos vêm dos mesmos sorteios.
- **`pct_e_replicas`** devolve duas coisas de uma vez (o % e os 200 % das réplicas), e o `pct_a, reps_a = ...` recebe as duas.
- **`gap - margem < 0 < gap + margem`** quer dizer "o zero está dentro do IC". O `not` inverte.

In [17]:
def pct_e_replicas(grupo, coluna_area):
    # As mesmas contas da estimativa_com_ic: o % com peso e os 200 % das réplicas
    y = grupo[coluna_area]
    peso = grupo['peso_amostral_individuo']
    pct = 100 * (peso * y).sum() / peso.sum()
    reps = grupo[colunas_rep]
    pct_reps = 100 * reps.mul(y, axis=0).sum() / reps.sum()
    return pct, pct_reps                     # devolve as duas coisas de uma vez


def gap_com_ic(responderam, coluna_area, ano, coluna_grupo, codigo_a, codigo_b):
    # Gap = % do grupo A menos % do grupo B (ex.: mais de 5 SM menos até 1 SM)
    pct_a, reps_a = pct_e_replicas(responderam[responderam[coluna_grupo] == codigo_a], coluna_area)
    pct_b, reps_b = pct_e_replicas(responderam[responderam[coluna_grupo] == codigo_b], coluna_area)

    gap = pct_a - pct_b                      # a diferença de verdade
    gaps_reps = reps_a - reps_b              # a diferença dentro de cada réplica: 200 números
    ep = gaps_reps.std() * fator[ano]        # quanto a diferença balança
    margem = 1.96 * ep

    return {'gap': round(float(gap), 1), 'ep': round(float(ep), 2),
            'ic_inf': round(float(gap - margem), 1), 'ic_sup': round(float(gap + margem), 1),
            'diferenca_real': not (gap - margem < 0 < gap + margem)}   # True = o IC não passa pelo zero

In [18]:
# Teste: 2024, renda (mais de 5 SM - até 1 SM) e região (Sudeste - Nordeste), em 3 áreas
d24 = base[base['ano'] == 2024]
linhas = []
for letra in ['E', 'B', 'C']:                                   # impostos, saúde e educação
    responderam = d24[d24[f'G2_{letra}'].notna()]
    for coluna_grupo, codigo_a, codigo_b, nome_par in [('renda5', 5, 1, 'mais de 5 SM - até 1 SM'),
                                                       ('regiao', 1, 2, 'Sudeste - Nordeste')]:
        linha = gap_com_ic(responderam, f'G2_{letra}', 2024, coluna_grupo, codigo_a, codigo_b)
        linha.update({'area': nomes_areas[letra], 'par': nome_par})
        linhas.append(linha)

pd.DataFrame(linhas)[['area', 'par', 'gap', 'ep', 'ic_inf', 'ic_sup', 'diferenca_real']]

,area,par,gap,ep,ic_inf,ic_sup,diferenca_real
0,impostos,mais de 5 SM - até 1 SM,31.5,7.55,16.7,46.3,True
1,impostos,Sudeste - Nordeste,21.8,7.00,8.1,35.5,True
2,saúde,mais de 5 SM - até 1 SM,11.2,10.48,-9.4,31.7,False
3,saúde,Sudeste - Nordeste,-0.1,5.94,-11.8,11.5,False
4,educação,mais de 5 SM - até 1 SM,-3.9,11.52,-26.4,18.7,False
5,educação,Sudeste - Nordeste,4.1,8.51,-12.5,20.8,False


Em impostos, as duas diferenças são reais. Em saúde e educação, nenhuma é. O gap negativo de educação 2024 (−3,9), que preocupava no 02, é só o balanço da amostra: o IC vai de −26,4 a 18,7.

Agora os 8 pares fixos da decisão nº 3, em todas as áreas e anos. Cada par tem 4 partes: a coluna, o código do grupo A, o do grupo B e um nome pra tabela.

In [19]:
# Decisão nº 3: os pares fixos (grupo A menos grupo B)
pares = [
    ('renda5', 5, 1, 'mais de 5 SM - até 1 SM'),
    ('escolaridade3', 3, 1, 'superior - até fundamental'),
    ('raca', 1, 2, 'branca - preta'),
    ('raca', 1, 3, 'branca - parda'),
    ('regiao', 1, 2, 'Sudeste - Nordeste'),
    ('regiao', 1, 3, 'Sudeste - Sul'),
    ('regiao', 1, 4, 'Sudeste - Norte'),
    ('regiao', 1, 5, 'Sudeste - Centro-Oeste'),
]

linhas = []
for ano in [2023, 2024, 2025]:
    d = base[base['ano'] == ano]
    for letra, nome_area in nomes_areas.items():
        if letra == 'H' and ano != 2025:                              # justiça só existe em 2025
            continue
        responderam = d[d[f'G2_{letra}'].notna()]
        for coluna_grupo, codigo_a, codigo_b, nome_par in pares:      # cada par tem 4 partes
            linha = gap_com_ic(responderam, f'G2_{letra}', ano, coluna_grupo, codigo_a, codigo_b)
            linha.update({'ano': ano, 'area': nome_area, 'dimensao': coluna_grupo, 'par': nome_par})
            linhas.append(linha)

gaps = pd.DataFrame(linhas)
gaps = gaps[['ano', 'area', 'dimensao', 'par', 'gap', 'ep', 'ic_inf', 'ic_sup', 'diferenca_real']]

# 22 combinações de ano e área × 8 pares = 176 linhas
assert gaps.shape == (176, 9), gaps.shape

# Quantos gaps são de verdade em cada par (de 22 possíveis). True vale 1 e False vale 0 na soma
print(gaps['diferenca_real'].sum(), 'de', len(gaps), 'gaps são de verdade\n')
print(gaps.groupby('par')['diferenca_real'].sum())

57 de 176 gaps são de verdade

par
Sudeste - Centro-Oeste         2
Sudeste - Nordeste            10
Sudeste - Norte                8
Sudeste - Sul                  5
branca - parda                 7
branca - preta                 2
mais de 5 SM - até 1 SM       11
superior - até fundamental    12
Name: diferenca_real, dtype: int64


In [20]:
# Os 10 maiores gaps
gaps.sort_values('gap', ascending=False).head(10)

,ano,area,dimensao,par,gap,ep,ic_inf,ic_sup,diferenca_real
41,2023,polícia,escolaridade3,superior - até fundamental,47.7,7.02,34.0,61.5,True
33,2023,impostos,escolaridade3,superior - até fundamental,45.3,5.90,33.8,56.9,True
17,2023,educação,escolaridade3,superior - até fundamental,39.9,9.53,21.2,58.6,True
32,2023,impostos,renda5,mais de 5 SM - até 1 SM,39.6,5.46,28.9,50.3,True
40,2023,polícia,renda5,mais de 5 SM - até 1 SM,35.1,9.18,17.2,53.1,True
54,2023,transporte,regiao,Sudeste - Norte,34.8,11.49,12.3,57.3,True
144,2025,impostos,renda5,mais de 5 SM - até 1 SM,33.8,7.64,18.9,48.8,True
80,2024,trabalho,renda5,mais de 5 SM - até 1 SM,32.9,8.19,16.8,49.0,True
88,2024,impostos,renda5,mais de 5 SM - até 1 SM,31.5,7.55,16.7,46.3,True
90,2024,impostos,raca,branca - preta,31.5,13.01,6.0,57.0,True


In [21]:
# Os gaps de verdade que são negativos: o grupo B resolve mais que o grupo A
gaps[gaps['ic_sup'] < 0]

,ano,area,dimensao,par,gap,ep,ic_inf,ic_sup,diferenca_real
79,2024,educação,regiao,Sudeste - Centro-Oeste,-20.9,9.10,-38.7,-3.0,True
101,2024,polícia,regiao,Sudeste - Sul,-40.5,13.65,-67.2,-13.7,True
109,2024,transporte,regiao,Sudeste - Sul,-24.9,11.32,-47.1,-2.7,True
125,2025,saúde,regiao,Sudeste - Sul,-9.8,4.78,-19.1,-0.4,True
132,2025,educação,regiao,Sudeste - Nordeste,-12.3,5.94,-23.9,-0.7,True
133,2025,educação,regiao,Sudeste - Sul,-31.6,6.85,-45.0,-18.2,True
157,2025,polícia,regiao,Sudeste - Sul,-25.2,8.36,-41.5,-8.8,True
159,2025,polícia,regiao,Sudeste - Centro-Oeste,-16.5,7.82,-31.9,-1.2,True


O que dá pra ver:
- **57 dos 176 gaps são de verdade.** Os pares com mais gaps reais são escolaridade (12 de 22) e renda (11 de 22). Branca − preta tem só 2: a categoria "preta" tem pesos muito desiguais, e o IC fica largo.
- **Os maiores gaps** são de escolaridade e de renda, quase todos em impostos e polícia, e a maioria em 2023.
- **O Sudeste nem sempre está no topo.** Os 8 gaps reais negativos são todos de região. O Sul resolve mais que o Sudeste em polícia, educação, transporte e saúde; o Centro-Oeste, em educação e polícia; e o Nordeste, em educação.
- **Cuidado:** com 176 gaps, alguns True podem ser sorte. De cada 20 gaps sem diferença nenhuma, mais ou menos 1 sai True por acaso. O teste das seções 9 e 10, com a correção de Holm, cuida disso.

## 9. O teste de Rao-Scott

**A ideia.** O gap compara 2 grupos. O teste compara todos os grupos de uma dimensão de uma vez e responde sim ou não: "a renda faz diferença em quem resolve online?"

- **Por que não o qui-quadrado comum:** ele faria a pergunta como se cada pessoa tivesse sido sorteada sozinha, todas com o mesmo peso. Não foi assim: a CETIC sorteia grupos de casas, e os pesos variam muito. Por isso a amostra "vale" bem menos gente do que o número de entrevistas (como o n efetivo do 02).
- **O Rao-Scott** mede, pelas réplicas, quanto a amostra vale menos (o efeito do desenho) e corrige o teste. É o de 2ª ordem, como foi decidido em 07/09.
- **O `svy`** faz a conta. Ele foi conferido contra a fórmula do pacote `survey` do R (o `svychisq`, com `statistic="F"`): nos 88 testes, deu exatamente o mesmo resultado.
- **O `scale`** faz o `svy` medir o balanço igual à gente: o desvio padrão das 200 réplicas (dividindo por 199), com o fator da decisão nº 9.
- **Raça nos testes:** só branca, preta e parda (decisão nº 4).
- **Como ler o p:** é a chance de aparecer uma diferença desse tamanho só por acaso, se o grupo não fizesse diferença nenhuma. Abaixo de 0,05 (o nosso α), a diferença é real.
- **`qui_quadrado_comum`** faz o teste errado, sem o desenho, só pra comparar.

In [22]:
def rao_scott(responderam, coluna_area, coluna_grupo, ano):
    # Teste de Rao-Scott (2ª ordem): a coluna_grupo faz diferença em resolver a área?
    tabela = responderam[[coluna_grupo, coluna_area, 'peso_amostral_individuo'] + colunas_rep]
    tabela = tabela[tabela[coluna_grupo].notna()]                    # tira quem não tem o grupo (ex.: renda NaN)
    dados_pl = pl.from_pandas(tabela.reset_index(drop=True))         # o svy usa polars, não pandas

    desenho = svy.Design(
        wgt='peso_amostral_individuo',
        rep_wgts=svy.BootstrapWgts(prefix='rep', n_reps=200, scale=fator[ano] ** 2 / 199),
    )
    amostra = svy.Sample(data=dados_pl, design=desenho)
    teste = amostra.categorical.tabulate(rowvar=coluna_grupo, colvar=coluna_area, method='replication').stats.f

    return {'F': round(teste.value, 2), 'gl1': round(teste.df_num, 2), 'gl2': round(teste.df_den, 1), 'p': teste.p_value}


def qui_quadrado_comum(dados_teste, coluna_area, coluna_grupo):
    # O qui-quadrado "comum", sem o desenho: só pra comparar com o Rao-Scott
    d = dados_teste[dados_teste[coluna_grupo].notna()]
    tabela = pd.crosstab(d[coluna_grupo], d[coluna_area], values=d['peso_amostral_individuo'], aggfunc='sum')
    tabela = tabela / tabela.to_numpy().sum() * len(d)          # os % com peso, em "número de pessoas" da amostra
    resultado = chi2_contingency(tabela, correction=False)
    return resultado.statistic, resultado.pvalue

In [23]:
# Teste: 3 exemplos de 2024, com o Rao-Scott e o qui-quadrado comum lado a lado
d24 = base[base['ano'] == 2024]
linhas = []
for nome_area, letra, coluna_grupo in [('impostos', 'E', 'renda5'), ('impostos', 'E', 'regiao'), ('saúde', 'B', 'regiao')]:
    responderam = d24[d24[f'G2_{letra}'].notna()]
    linha = rao_scott(responderam, f'G2_{letra}', coluna_grupo, 2024)
    x2_comum, p_comum = qui_quadrado_comum(responderam, f'G2_{letra}', coluna_grupo)
    linha.update({'area': nome_area, 'dimensao': coluna_grupo, 'x2_comum': round(x2_comum, 1), 'p_comum': p_comum})
    linhas.append(linha)

pd.DataFrame(linhas)[['area', 'dimensao', 'F', 'gl1', 'gl2', 'p', 'x2_comum', 'p_comum']]

,area,dimensao,F,gl1,gl2,p,x2_comum,p_comum
0,impostos,renda5,2.86,2.81,558.7,0.039904,100.2,9.045151e-21
1,impostos,regiao,6.24,2.14,425.5,0.001679,153.0,4.542239e-32
2,saúde,regiao,0.42,2.01,399.4,0.657950,10.1,3.826145e-02


O que dá pra ver:
- **Impostos 2024, por renda:** o qui-quadrado comum dá 100,2, com p perto de zero ("certeza absoluta"). O Rao-Scott dá p = 0,04: ainda é diferença real, mas por pouco. O efeito do desenho aqui é de umas 9 vezes (100,2 ÷ 2,86 ÷ 4, em que o 4 são as 5 faixas menos 1): é como se a amostra valesse 9 vezes menos gente do que parece.
- **Saúde 2024, por região:** até aqui o qui-quadrado comum diria que existe diferença (p = 0,038). O Rao-Scott diz que não (p = 0,66).
- **Impostos 2024, por região:** p = 0,0017, diferença real. Com a saúde (p = 0,66, nenhuma diferença), os dois batem com os gaps Sudeste − Nordeste da seção 8 (+21,8 real e −0,1).

Agora os 88 testes: 7 áreas × 4 dimensões × 3 anos, mais as 4 da justiça em 2025. É o mesmo esqueleto do tabelão; na raça, o `if` fica só com branca, preta e parda (`None` quer dizer "sem filtro"). Leva uns 10 segundos.

In [24]:
# As 4 dimensões dos testes. Na raça, só branca, preta e parda (decisão nº 4)
dimensoes = {'renda5': None, 'escolaridade3': None, 'raca': [1, 2, 3], 'regiao': None}

linhas = []
for ano in [2023, 2024, 2025]:
    d = base[base['ano'] == ano]
    for letra, nome_area in nomes_areas.items():
        if letra == 'H' and ano != 2025:                                    # justiça só existe em 2025
            continue
        responderam = d[d[f'G2_{letra}'].notna()]
        for coluna_grupo, codigos in dimensoes.items():
            dados_teste = responderam
            if codigos is not None:                                         # só na raça: fica com 1, 2 e 3
                dados_teste = responderam[responderam[coluna_grupo].isin(codigos)]
            linha = rao_scott(dados_teste, f'G2_{letra}', coluna_grupo, ano)
            linha['p_comum'] = qui_quadrado_comum(dados_teste, f'G2_{letra}', coluna_grupo)[1]   # [1] = só o p
            linha.update({'ano': ano, 'area': nome_area, 'dimensao': coluna_grupo})
            linhas.append(linha)

testes = pd.DataFrame(linhas)
testes = testes[['ano', 'area', 'dimensao', 'F', 'gl1', 'gl2', 'p', 'p_comum']]
print(testes.shape)

(88, 8)


## 10. A correção de Holm

Com muitos testes, alguns dão p abaixo de 0,05 por sorte. Num ano com 28 testes, mesmo que nada fizesse diferença, uns 1 ou 2 passariam de 0,05 por acaso. O Holm deixa a régua mais exigente: o menor p é multiplicado pelo número de testes (28), o segundo por 27, e assim por diante.

- **`cummax`** guarda o maior valor até ali: um p ajustado nunca pode ser menor que o anterior na fila. O **`clip(upper=1)`** corta no 1.
- **O brinquedo:** 0,01 × 3 = 0,03; 0,03 × 2 = 0,06; 0,04 × 1 = 0,04, mas não pode ser menor que 0,06, então vira 0,06. O 0,04 era real (abaixo de 0,05) e deixou de ser.
- **Os grupos:** o Holm é aplicado separado em cada ano (28 testes), e a justiça fica à parte (decisão nº 5). O `transform(holm)` faz isso.

In [25]:
def holm(p_valores):
    # Correção de Holm: o menor p é multiplicado pelo número de testes (m), o segundo por m - 1, e assim por diante
    p = pd.Series(p_valores).sort_values()               # do menor pro maior
    m = len(p)
    ajustado = p * np.arange(m, 0, -1)                    # multiplica por m, m - 1, ..., 1
    ajustado = ajustado.cummax().clip(upper=1)            # nunca diminui na ordem, e no máximo 1
    return ajustado.sort_index()                          # volta pra ordem original


# Teste de brinquedo com 3 p-valores
holm([0.01, 0.04, 0.03])

0    0.03
1    0.06
2    0.06
dtype: float64

In [26]:
# Holm por ano, com justiça à parte (decisão nº 5)
testes['grupo_holm'] = testes['ano'].astype(str)
testes.loc[testes['area'] == 'justiça', 'grupo_holm'] = 'justiça'
testes['p_holm'] = testes.groupby('grupo_holm')['p'].transform(holm)
testes['real_05'] = testes['p'] < 0.05          # real antes do Holm
testes['real_holm'] = testes['p_holm'] < 0.05   # real depois do Holm

assert testes.shape[0] == 88, testes.shape

# Quantos testes dão diferença real em cada régua
funil = pd.DataFrame({
    'qui_quadrado_comum': (testes['p_comum'] < 0.05).groupby(testes['grupo_holm']).sum(),
    'rao_scott': testes.groupby('grupo_holm')['real_05'].sum(),
    'rao_scott_holm': testes.groupby('grupo_holm')['real_holm'].sum(),
})
funil.loc['total'] = funil.sum()
funil

,qui_quadrado_comum,rao_scott,rao_scott_holm
grupo_holm,,,
2023,28,14,5
2024,26,10,1
2025,25,11,4
justiça,4,1,0
total,83,36,10


In [27]:
# Os testes que continuam reais depois do Holm
testes[testes['real_holm']].sort_values('p_holm')

,ano,area,dimensao,F,gl1,gl2,p,p_comum,grupo_holm,p_holm,real_05,real_holm
21,2023,polícia,escolaridade3,17.02,1.88,374.6,1.729428e-07,2.372779e-37,2023,0.000005,True,True
17,2023,impostos,escolaridade3,16.17,1.61,320.8,1.891147e-06,4.845924e-69,2023,0.000051,True,True
67,2025,educação,regiao,8.45,3.28,652.4,7.830881e-06,4.973287e-35,2025,0.000219,True,True
75,2025,impostos,regiao,7.76,2.89,575.3,5.614089e-05,5.576752e-30,2025,0.001516,True,True
72,2025,impostos,renda5,6.57,3.64,723.8,6.644597e-05,2.542636e-40,2025,0.001728,True,True
73,2025,impostos,escolaridade3,10.43,1.83,363.6,7.149301e-05,6.322250e-29,2025,0.001787,True,True
9,2023,educação,escolaridade3,10.56,1.75,347.4,8.618972e-05,5.795794e-62,2023,0.002241,True,True
13,2023,trabalho,escolaridade3,6.92,1.88,374.8,1.420925e-03,1.476655e-56,2023,0.035523,True,True
19,2023,impostos,regiao,5.05,3.15,626.6,1.519977e-03,8.784449e-20,2023,0.036479,True,True
47,2024,impostos,regiao,6.24,2.14,425.5,1.679261e-03,4.542239e-32,2024,0.047019,True,True


O que dá pra ver:
- **O funil:** o qui-quadrado comum diria que 83 dos 88 testes têm diferença real. O Rao-Scott, 36. Depois do Holm, sobram 10. Não é erro: é o que a amostra permite afirmar com segurança.
- **Os 10 que sobram:** 5 de escolaridade, 4 de região, 1 de renda e nenhum de raça. Seis dos 10 são de impostos.
- **Não passar no Holm não quer dizer que a diferença não existe.** Quer dizer que a amostra não tem força pra afirmar. Exemplo: impostos 2024 por renda tinha p = 0,04 e um gap real nas pontas (IC de 16,7 a 46,3), mas o teste das 5 faixas juntas não passa no Holm.

## 11. O tamanho da diferença: V de Cramér

O p diz **se** existe diferença. O V de Cramér diz **o tamanho** dela: 0 quer dizer nenhuma diferença, e 1, a maior possível. Pela régua de Cohen, 0,1 é pequena, 0,3 é média e 0,5 é grande.

- A conta usa a mesma tabela "grupo × resolveu" do teste, mas somando os pesos em vez de contar pessoas: quantos brasileiros cada casa da tabela representa. O tamanho da diferença é sobre a população, então não depende do desenho da amostra.
- O `scipy` só aceita números inteiros na tabela. Os pesos somam milhões, então arredondar não muda nada.
- No fim, o `merge` junta o V na tabela `testes`. O `drop(..., errors='ignore')` tira um V antigo, se a célula já tiver rodado antes.

In [28]:
def v_cramer(dados_teste, coluna_area, coluna_grupo):
    # Tamanho da diferença entre os grupos: 0 = nenhuma, 1 = a maior possível
    d = dados_teste[dados_teste[coluna_grupo].notna()]
    tabela = pd.crosstab(d[coluna_grupo], d[coluna_area], values=d['peso_amostral_individuo'], aggfunc='sum')   # tabela com os pesos
    tabela = tabela.round().astype(int)       # o scipy só aceita inteiros (os pesos somam milhões, arredondar não muda nada)
    return round(association(tabela.to_numpy(), method='cramer'), 3)


# Teste: impostos 2024, por renda
impostos_2024 = d24[d24['G2_E'].notna()]
v_cramer(impostos_2024, 'G2_E', 'renda5')

0.192

In [29]:
linhas = []
for ano in [2023, 2024, 2025]:
    d = base[base['ano'] == ano]
    for letra, nome_area in nomes_areas.items():
        if letra == 'H' and ano != 2025:
            continue
        responderam = d[d[f'G2_{letra}'].notna()]
        for coluna_grupo, codigos in dimensoes.items():
            dados_teste = responderam
            if codigos is not None:
                dados_teste = responderam[responderam[coluna_grupo].isin(codigos)]
            linhas.append({'ano': ano, 'area': nome_area, 'dimensao': coluna_grupo,
                           'v_cramer': v_cramer(dados_teste, f'G2_{letra}', coluna_grupo)})

efeitos = pd.DataFrame(linhas)
testes = testes.drop(columns='v_cramer', errors='ignore')     # se a célula já rodou antes, tira o V antigo
testes = testes.merge(efeitos, on=['ano', 'area', 'dimensao'], how='left', validate='one_to_one')
print(testes.shape)

(88, 13)


## 12. O ranking das áreas

O ranking do projeto é pelo tamanho da diferença (o V), e não pelo p: em cada ano e dimensão, as 7 áreas vão da maior diferença (1º lugar) pra menor (7º). A justiça fica sem ranking, porque só existe em 2025.

- **`rank(ascending=False)`** dá o lugar de cada área (o maior V fica com 1). O resultado vem com vírgula (1.0, 2.0...), e o **`Int64`** (com I maiúsculo) vira número inteiro, aceitando o vazio da justiça.
- **`unstack()`** vira as dimensões em colunas, como um `pivot`.
- **O ranking sempre se lê junto com o `real_holm`:** o tamanho e a certeza são coisas diferentes (o exemplo de região em 2024 mostra isso).

In [30]:
# Ranking: em cada ano e dimensão, as 7 áreas da maior diferença (1º) pra menor (7º). Justiça fica sem ranking
sem_justica = testes['area'] != 'justiça'
testes['ranking'] = testes[sem_justica].groupby(['ano', 'dimensao'])['v_cramer'].rank(ascending=False)
testes['ranking'] = testes['ranking'].astype('Int64')          # número inteiro, com vazio na justiça

# O V médio de cada dimensão, por ano (o unstack vira as dimensões em colunas)
print(testes[sem_justica].groupby(['ano', 'dimensao'])['v_cramer'].mean().unstack().round(3), '\n')

# O V médio dos 3 anos juntos, por dimensão e por área
print(testes[sem_justica].groupby('dimensao')['v_cramer'].mean().round(3).sort_values(ascending=False), '\n')
print(testes[sem_justica].groupby('area')['v_cramer'].mean().round(3).sort_values(ascending=False), '\n')

# Quantas vezes cada área ficou em 1º lugar (de 12: 3 anos × 4 dimensões)
print(testes[testes['ranking'] == 1]['area'].value_counts())

dimensao  escolaridade3   raca  regiao  renda5
ano                                           
2023              0.236  0.148   0.181   0.192
2024              0.096  0.135   0.193   0.186
2025              0.116  0.102   0.151   0.119 

dimensao
regiao           0.175
renda5           0.166
escolaridade3    0.149
raca             0.128
Name: v_cramer, dtype: float64 

area
impostos      0.201
polícia       0.195
transporte    0.182
educação      0.158
trabalho      0.118
documentos    0.116
saúde         0.111
Name: v_cramer, dtype: float64 

area
polícia       4
impostos      4
transporte    3
educação      1
Name: count, dtype: int64


In [31]:
# Exemplo: o ranking de região em 2024
testes[(testes['ano'] == 2024) & (testes['dimensao'] == 'regiao')].sort_values('ranking')

,ano,area,dimensao,F,gl1,gl2,p,p_comum,grupo_holm,p_holm,real_05,real_holm,v_cramer,ranking
51,2024,polícia,regiao,5.41,2.30,457.8,0.003103,5.603907e-30,2024,0.083782,True,False,0.347,1
47,2024,impostos,regiao,6.24,2.14,425.5,0.001679,4.542239e-32,2024,0.047019,True,True,0.230,2
55,2024,transporte,regiao,3.48,2.04,405.1,0.031094,1.720818e-12,2024,0.652971,True,False,0.219,3
39,2024,educação,regiao,3.13,2.21,440.1,0.039602,9.483410e-20,2024,0.792031,True,False,0.200,4
43,2024,trabalho,regiao,4.53,2.59,515.1,0.005907,7.749256e-20,2024,0.153591,True,False,0.173,5
31,2024,documentos,regiao,2.25,2.60,518.3,0.090668,1.019787e-11,2024,1.000000,False,False,0.131,6
35,2024,saúde,regiao,0.42,2.01,399.4,0.657950,3.826145e-02,2024,1.000000,False,False,0.051,7


O que dá pra ver:
- **Dimensões:** pelo V médio dos 3 anos, região (0,175) e renda (0,166) têm as maiores diferenças, depois escolaridade (0,149) e raça (0,128). As quatro ficam na faixa "pequena" da régua de Cohen. Em 2023 a maior era escolaridade (0,236), que caiu em 2024 e 2025 (0,096 e 0,116).
- **Áreas:** impostos (0,201), polícia (0,195) e transporte (0,182) são as mais desiguais, e juntas ficam com 11 dos 12 primeiros lugares. Saúde é a mais igual (0,111).
- **Tamanho não é certeza:** em região 2024, o 1º lugar é polícia, com V de 0,347 (o maior do ano), mas ele não passa no Holm (p ajustado de 0,084): polícia tem pouca gente respondendo (1.191 pessoas) e pesos muito desiguais. O 2º lugar, impostos (0,230), passa.

## 13. Testes de integridade

Igual aos notebooks 01 e 02: antes de salvar, conferir se as tabelas estão certas. Se algum `assert` falhar, o notebook para com erro e diz o que deu errado.

1. **Tamanhos:** 418, 176 e 88 linhas.
2. **Nada impossível:** nenhum grupo vazio, p e V entre 0 e 1, e o Holm nunca diminui o p.
3. **O IC sempre envolve a estimativa:** o % fica entre o `ic_inf` e o `ic_sup` (e o gap também).
4. **O ranking:** em cada ano e dimensão, as 7 áreas têm os lugares de 1 a 7, sem repetir.

In [32]:
# 1) Tamanhos previstos
assert tabelao.shape == (418, 11), tabelao.shape
assert gaps.shape == (176, 9), gaps.shape
assert testes.shape == (88, 14), testes.shape

# 2) Nada vazio ou impossível
assert (tabelao['n'] > 0).all(), 'Tem grupo vazio no tabelão'
assert testes['p'].between(0, 1).all() and testes['p_holm'].between(0, 1).all(), 'p fora de 0 a 1'
assert (testes['p_holm'] >= testes['p']).all(), 'O Holm diminuiu algum p'
assert testes['v_cramer'].between(0, 1).all(), 'V fora de 0 a 1'

# 3) O IC sempre envolve a estimativa
assert ((tabelao['ic_inf'] <= tabelao['pct']) & (tabelao['pct'] <= tabelao['ic_sup'])).all(), 'IC do tabelão errado'
assert ((gaps['ic_inf'] <= gaps['gap']) & (gaps['gap'] <= gaps['ic_sup'])).all(), 'IC dos gaps errado'

# 4) Ranking: em cada ano e dimensão, as 7 áreas têm os lugares 1 a 7, sem repetir
lugares = testes[testes['area'] != 'justiça'].groupby(['ano', 'dimensao'])['ranking'].apply(sorted)
assert all(lista == list(range(1, 8)) for lista in lugares), 'Ranking com lugar repetido ou faltando'

print('Todos os testes passaram')

Todos os testes passaram


## 14. Salvar

- As 3 tabelas vão pra `outputs/`, no formato do Power BI em português, igual ao 02: `sep=';'`, `decimal=','` e `encoding='utf-8-sig'` (pro Excel abrir os acentos).
- **`float_format='%.6f'`** (só nos testes): escreve os números com 6 casas, sem a notação científica (`1,7e-07`), que o Power BI pode ler errado. Um p de 0,000000 quer dizer "menor que 0,000001".
- O que tem em cada arquivo está no `outputs/README.md`.

In [33]:
pasta_outputs = os.path.join(raiz_projeto, 'outputs')

tabelao.to_csv(os.path.join(pasta_outputs, 'estimativas_ic.csv'), sep=';', decimal=',', index=False, encoding='utf-8-sig')
gaps.to_csv(os.path.join(pasta_outputs, 'gaps_ic.csv'), sep=';', decimal=',', index=False, encoding='utf-8-sig')
testes.to_csv(os.path.join(pasta_outputs, 'testes_rao_scott.csv'), sep=';', decimal=',', index=False, encoding='utf-8-sig',
              float_format='%.6f')

for nome, tabela in [('estimativas_ic.csv', tabelao), ('gaps_ic.csv', gaps), ('testes_rao_scott.csv', testes)]:
    print('Salvo: outputs/' + nome, '|', tabela.shape[0], 'linhas x', tabela.shape[1], 'colunas')

Salvo: outputs/estimativas_ic.csv | 418 linhas x 11 colunas
Salvo: outputs/gaps_ic.csv | 176 linhas x 9 colunas
Salvo: outputs/testes_rao_scott.csv | 88 linhas x 14 colunas


## 15. Resumo dos achados

1. **Os nossos números conferem com os da CETIC.** Com o universo dos usuários de Internet de 16 anos ou mais, o G1 de 2024 bate exatamente com o oficial. As margens de 2024 e 2025 são as nossas × 1,054 e × 1,14, um fator constante que a gente aplica (decisão nº 9); as de 2023 ficam perto das oficiais.
2. **A amostra vale bem menos do que o número de entrevistas.** O qui-quadrado comum diria que 83 dos 88 testes têm diferença real. O Rao-Scott, que leva em conta o desenho da amostra, diz 36. Depois da correção de Holm, sobram 10. Usar o Rao-Scott, e não o qui-quadrado simples, é uma melhoria de método em relação a estudos anteriores com a mesma base.
3. **As diferenças são pequenas, mas existem.** Pelo V de Cramér médio, a ordem é região (0,175), renda (0,166), escolaridade (0,149) e raça (0,128), todas na faixa "pequena" da régua de Cohen. Escolaridade era a maior em 2023 (0,236) e caiu em 2024 e 2025.
4. **Escolaridade e região são as diferenças mais firmes.** Dos 10 testes que passam no Holm, 5 são de escolaridade e 4 de região. Renda tem 1, e raça nenhuma.
5. **Impostos é a área mais desigual e a mais certa.** O gap de renda (mais de 5 SM menos até 1 SM) é de 39,6, 31,5 e 33,8 pontos em 2023, 2024 e 2025, todos reais, e 6 dos 10 testes que passam no Holm são de impostos. Polícia e transporte também têm diferenças grandes, mas pouca gente respondendo, e quase nunca passam no Holm.
6. **Saúde é a área mais igual.** Tem o menor V médio (0,111), e em 2024 a região não faz diferença nenhuma (p = 0,66).
7. **O Sudeste nem sempre está no topo.** Os 8 gaps reais negativos são de região: o Sul resolve mais que o Sudeste em polícia, educação, transporte e saúde (pelos ICs dos gaps, sem a correção de Holm).
8. **Raça: nenhum teste passa no Holm.** Isso não prova que não existe desigualdade racial. A categoria "preta" tem n efetivo baixo em várias áreas, e amarela e indígena ficaram fora dos testes. É uma limitação da amostra, que vai pro texto do método.
9. **Os gaps negativos do 02 eram balanço da amostra.** Educação 2024, por renda: −3,9 pontos, com IC de −26,4 a 18,7.

Arquivos salvos em `outputs/`: `estimativas_ic.csv`, `gaps_ic.csv` e `testes_rao_scott.csv`.